# 66. 화면 답변 평가 정책과 기존 결과 재검토

API 호출 없이 기존 답변을 직접요구·화면사실정확성·입력지원·추가설명으로 분리한다. 재채점 상승은 모델 개선이 아니다.

공통 helper는 기존 결과 결속, 수동판정 검사, 완전행렬 집계를 재사용하기 위해 scripts/에 둔다. HTML은 결과 확인 후 직접 작성한다.

In [1]:
from pathlib import Path
import importlib.util, json, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
spec = importlib.util.spec_from_file_location('visible66_notebook', ROOT/'notebooks/scripts/66_visible_answer_evaluation.py')
E = importlib.util.module_from_spec(spec); spec.loader.exec_module(E)
BASE = E.BASE
policy = E.read(BASE/'policy.json')
rubrics = E.read(BASE/'question_contract.json')
print('질문', len(rubrics['records']), 'API 호출', policy['API_calls'])
context = E.read(BASE/'evaluation_context.json')
assert context['artifact_roles'] == E.expected_roles(66)
print('평가 문맥 파일 검증', E.verify_file_map(context['files_sha256']))

질문 48 API 호출 0
평가 문맥 파일 검증 2022


In [2]:
# 원본이 보존되어 있으면 검토용 캐시는 언제든 다시 만들 수 있다.
from collections import defaultdict
source_by_query = defaultdict(list)
for item in E.read(ROOT/context['artifact_roles']['source_registry'])['records']:
    source_by_query[item['query_id']].append(item)
answer_by_id = {r['identity']: r['answer'] for r in E.read(E.UPSTREAM/'answer_results.json')['records']}
all_jobs = E.read(E.UPSTREAM/'jobs.json')['records']
cache_dir = BASE/'question_packets'; cache_dir.mkdir(exist_ok=True)
for rubric in rubrics['records']:
    qid = rubric['query_id']
    records = [{'job': j, 'answer_row': answer_by_id[j['identity']], 'binding': E.binding(j, answer_by_id[j['identity']], rubric, context)} for j in all_jobs if j['query_id'] == qid]
    refs = sorted({(c['card_key'], f['condition_id']) for c in source_by_query[qid] for f in c['facts'] if f['source_status'] == 'verified'})
    packet = {'status': 'source_and_saved_output_packet_not_new_judgment', 'query_id': qid, 'rubric': rubric, 'records': records, 'source_records': source_by_query[qid], 'evaluation_context_sha256': E.digest(context), 'allowed_source_refs': [{'card_key': k, 'condition_id': f} for k, f in refs], 'instructions': ['평가 대상은 저장된 본문과 카드별 추천 이유뿐이다.', '이 패킷은 새 채점이 아니다. 직접 요구·실제 주장·입력 지원·추가 설명을 수동으로 분리한다.', 'source_records의 조건은 원문 사실의 참조 자료이며 모든 조건이 질문의 필수 요구인 것은 아니다.', '기존 입력에 없어도 원문상 정확한 주장은 정답성과 입력 지원을 별도로 기록한다.']}
    target = cache_dir/(qid+'.json')
    if target.exists():
        assert E.read(target) == packet, '검토 캐시가 고정 입력과 다릅니다: '+qid
    else:
        target.write_text(json.dumps(packet, ensure_ascii=False, indent=2)+'\n')
print('검토 캐시', len(rubrics['records']))

검토 캐시 48


## 수동 의미 검토

질문별 직접 요구는 question_contract.json에 사전 고정한다. 판정은 data/66_visible_answer_policy/reviews/에 저장하며 실제 화면·원문·실제 입력을 읽는다. helper가 정답을 추론하지 않는다. 미완료 행을 통과로 채우지 않는다.

In [3]:
review_files = sorted((BASE/'reviews').glob('*.json'))
reviewed = sum(len(E.read(p)['records']) for p in review_files)
print('검토 저장 행', reviewed, '/ 576')
assert reviewed <= 576, '중복 판정 파일을 확인하세요'

검토 저장 행 576 / 576


## 최종 집계

저장된 576개 판정을 실제 화면·원문·입력 결속으로 다시 검사하고 완전 행렬만 집계한다. `claims`는 화면 성능을 보충하지 않는다. 아래 점수의 변화는 같은 답변을 다른 정책으로 재채점한 결과이며 모델 개선이 아니다. 추가 설명 충실도는 각 검토 행의 정성 메모로 남긴다.

In [4]:
import time, resource
from collections import Counter
started=time.perf_counter();cpu_started=time.process_time()
all_scores=[];review_hashes={}
for rubric in rubrics['records']:
    packet=E.read(BASE/'question_packets'/(rubric['query_id']+'.json'))
    path=BASE/'reviews'/(rubric['query_id']+'.json')
    document=E.read(path)
    reviewed={r['identity']:r for r in document['records']}
    assert len(reviewed)==12
    allowed={(r['card_key'],r['condition_id'])for r in packet['allowed_source_refs']}
    for item in packet['records']:
        identity=item['job']['identity']
        all_scores.append(E.score_review(item['job'],item['answer_row'],rubric,reviewed[identity],allowed,context=context))
    review_hashes[str(path.relative_to(ROOT))]=E.file_digest(path)
    if len(all_scores)%144==0:print('최종 재검사',len(all_scores),'/576')
profiles=['card_page_section_benefit','parent_child_bundle'];modes=['U','SEQ'];repeats=[1,2,3]
summary=E.aggregate(all_scores,query_ids=[r['query_id']for r in rubrics['records']],profiles=profiles,modes=modes,repeats=repeats,context=context)
groups={}
for group in ('explicit','general','uncertain'):
    qids=[r['query_id']for r in rubrics['records']if r['group']==group]
    groups[group]=E.aggregate([r for r in all_scores if r['query_id']in qids],query_ids=qids,profiles=profiles,modes=modes,repeats=repeats,context=context)
old=E.read(E.UPSTREAM/'manual_judgments.json')['records'];old_by_id={r['identity']:r for r in old}
policy_changes=[];paired=[]
for profile in profiles:
    for mode in modes:
        selected=[r for r in all_scores if r['profile']==profile and r['mode']==mode]
        changes=[{'identity':r['identity'],'old_legacy_pass':old_by_id[r['identity']]['legacy_pass'],'new_visible_pass':r['passed'],
                  'visible_errors':r['visible_errors'],'missing_requirement_ids':r['missing_requirement_ids']}
                 for r in selected if old_by_id[r['identity']]['legacy_pass']!=r['passed']]
        policy_changes.append({'profile':profile,'mode':mode,'old_pass_count':sum(old_by_id[r['identity']]['legacy_pass']for r in selected),
            'new_pass_count':sum(r['passed']for r in selected),'same_saved_answers':True,'model_improvement':False,'changes':changes})
    bymode={x['mode']:x for x in summary if x['profile']==profile};wins=[];losses=[];ties=[]
    for qid in bymode['U']['per_query_pass_count']:
        a=bymode['U']['per_query_pass_count'][qid];b=bymode['SEQ']['per_query_pass_count'][qid]
        (wins if b>a else losses if b<a else ties).append({'query_id':qid,'U_pass_count':a,'SEQ_pass_count':b})
    paired.append({'profile':profile,'wins':wins,'losses':losses,'ties':ties,'unit':'동일 질문의 3회 통과 개수 비교'})
wording=E.read(BASE/'wording_adjudication.json')
sensitivity=[]
for row in summary:
    flagged=[r for r in all_scores if r['profile']==row['profile'] and r['mode']==row['mode'] and r['identity']in wording['sensitivity_uncertain_identities'] and r['passed']]
    sensitivity.append({'profile':row['profile'],'mode':row['mode'],'primary_passed':row['passed'],'conservative_wording_passed':row['passed']-len(flagged),'denominator':row['logical_answers'],'flagged_identities':[r['identity']for r in flagged]})
result={'status':'complete_matrix_scored_pending_independent_acceptance','evaluation_context_sha256':E.digest(context),
    'policy_version':E.POLICY_VERSION,'records':all_scores,'summary':summary,'groups':groups,
    'wording_sensitivity':sensitivity,'wording_adjudication_raw_sha256':E.file_digest(BASE/'wording_adjudication.json'),
    'policy_change_comparison':policy_changes,'U_SEQ_query_pairs':paired,
    'review_files_sha256':review_hashes,'new_API_calls':0,'new_embeddings':0,
    'operating_code_changed':False,'production_adopted':False,
    'limitation':'같은 저장답변의 정책 변경 효과다. 최적화된 서로 다른 운영프로필 전체 비교이며 청킹 단독 인과효과나 신규 독립확인이 아니다.'}
def persist_evaluation(name,value):
    path=BASE/name
    if path.exists():assert E.read(path)==value,'이전 집계와 다름: '+name
    else:path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n')
persist_evaluation('evaluation_results.json',result)
resources={'status':'offline_full_576_review_validation','wall_seconds':time.perf_counter()-started,
    'cpu_seconds':time.process_time()-cpu_started,'process_peak_RSS_KiB':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    'new_API_requests':0,'new_API_cost_USD':0,'new_embeddings':0,'GPU':'미사용',
    'human_semantic_review_time':'미측정','browser_display_seconds':'미측정',
    'scope':'위 시간은 로컬 재채점 검증·집계 시간이다. 검색·답변 생성·화면 출력의 온라인 지연이 아니다.'}
resource_path=BASE/'final_scoring_resources.json'
if not resource_path.exists():resource_path.write_text(json.dumps(resources,ensure_ascii=False,indent=2)+'\n')
for row in summary:print(row['profile'],row['mode'],row['passed'],'/',row['logical_answers'],round(row['pass_macro']*100,2),'%')
print('새 API 비용 $0 / 운영반영 없음 / 독립 검토 대기')

최종 재검사 144 /576
최종 재검사 288 /576
최종 재검사 432 /576
최종 재검사 576 /576
card_page_section_benefit U 128 / 144 88.89 %
card_page_section_benefit SEQ 134 / 144 93.06 %
parent_child_bundle U 120 / 144 83.33 %
parent_child_bundle SEQ 123 / 144 85.42 %
새 API 비용 $0 / 운영반영 없음 / 독립 검토 대기
